# Virtual Screening — HsDHODH Inhibitors (PyCaret)

**Objective:** Perform Virtual Screening using Machine Learning models (Random Forest and XGBoost) trained with ECFP fingerprints to identify potential HsDHODH inhibitors.

## Notebook Pipeline

1. **Configuration** — Model paths, data paths, and parameters
2. **Virtual Screening with Applicability Domain (AD)** — Predictions + applicability domain assessment
3. **Interpretability (SHAP)** — Prediction explanations [Optional]

---

## Technical Information

- **Models:** Random Forest and XGBoost (trained with PyCaret)
- **Fingerprints:** ECFP (diameter 4, 1845 bits after feature selection)
- **Applicability Domain:** Tanimoto Distance (k-NN, k=5)
- **AD Threshold:** 0.7 (distance > 0.7 = outside domain)

## 1. Configuration

Adjust model paths, input/training data paths, and Applicability Domain parameters.

In [9]:
from pathlib import Path

# ============================================================================
# CONFIGURATION - Adjust as needed
# ============================================================================

# Project base directory
BASE_PATH = Path(r"C:\Users\sabri\repositorios\Identification-of-Potent-HsDHODH-Inhibitors-via-Integrated-Computational-Pipeline")

# Trained model paths (PyCaret)
MODEL_PATHS = [
    Path(r"C:\Users\sabri\OneDrive\Documentos\UFG\project-BRICS\HsDHODH\QSAR-ML\ML-pycaret\models\HsDHODH_ECFP_diam4_1845bits_RandomForest_tunned"),
    Path(r"C:\Users\sabri\OneDrive\Documentos\UFG\project-BRICS\HsDHODH\QSAR-ML\ML-pycaret\models\HsDHODH_ECFP_diam4_1845bits_XGBoost_tunned"),
]

# TRAINING data (required for Applicability Domain calculation)
TRAIN_DATA_PATH = Path(r"C:\Users\sabri\OneDrive\Documentos\UFG\project-BRICS\HsDHODH\QSAR-ML\ML-pycaret\fp-gen\ECFP\HsDHODH-curated_ecfp_diam4_2048bits.csv")

# SCREENING data (compounds to be predicted)
CSV_PATH = BASE_PATH / "fp-gen/ECFP/diam4/ecfp_diam4_CPDS.csv"

# Output directory
OUTPUT_DIR = BASE_PATH / "output"

# Identification columns
SMILES_COL = "SMILES"
MOL_ID_COL = "ID"
FINGERPRINT_PREFIX = "bit_"

# ============================================================================
# APPLICABILITY DOMAIN PARAMETERS
# ============================================================================
# Tanimoto distance threshold for AD
# If distance > threshold → outside applicability domain
TANIMOTO_THRESHOLD = 0.7

# Number of neighbors for k-NN
K_NEIGHBORS = 5

print("✅ Configuration loaded!")
print(f"   📁 Models: {len(MODEL_PATHS)}")
print(f"   📁 Training data: {TRAIN_DATA_PATH.name}")
print(f"   📁 Screening data: {CSV_PATH.name}")
print(f"   📊 AD Threshold: {TANIMOTO_THRESHOLD}")

✅ Configuration loaded!
   📁 Models: 2
   📁 Training data: HsDHODH-curated_ecfp_diam4_2048bits.csv
   📁 Screening data: ecfp_diam4_CPDS.csv
   📊 AD Threshold: 0.7


## 2. Virtual Screening with Applicability Domain

This cell executes the complete pipeline:
1. Loads models and extracts the **1845 selected features**
2. Loads screening and training data
3. Calculates **Tanimoto Distance** (AD) using selected features
4. Performs **predictions** with RF and XGBoost
5. Calculates **consensus** (mean of probabilities)
6. Saves results to CSV

In [10]:
import pandas as pd
import numpy as np
import datetime as dt
import sys
from pycaret.classification import load_model, predict_model
from scipy.spatial.distance import cdist

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_model_safe(path: Path):
    """Safely load PyCaret model."""
    try:
        return load_model(str(path.with_suffix("")))
    except Exception as e:
        print(f"❌ Error loading {path.name}: {e}")
        sys.exit(1)


def get_selected_features_from_pipeline(model):
    """
    Extract SELECTED features from PyCaret pipeline.
    Features are in the last step (trained_model), which is the estimator trained
    AFTER multicollinearity removal.
    """
    if hasattr(model, 'steps'):
        final_estimator = model.steps[-1][1]
        if hasattr(final_estimator, 'feature_names_in_'):
            return list(final_estimator.feature_names_in_)
    
    if hasattr(model, 'feature_names_in_'):
        return list(model.feature_names_in_)
    
    return None


def calculate_ad_tanimoto(query_matrix, train_matrix, k_neighbors=5):
    """
    Calculate Applicability Domain using Tanimoto (Jaccard) distance.
    
    For each query compound, calculates the mean distance to the k nearest 
    neighbors in the training set.
    
    Tanimoto Distance = 1 - Tanimoto Similarity
    """
    print(f"   -> Calculating Tanimoto distance (Query: {query_matrix.shape}, Train: {train_matrix.shape})...")
    
    # cdist with 'jaccard' calculates Jaccard distance (1 - Tanimoto)
    dists = cdist(query_matrix, train_matrix, metric='jaccard')
    
    # Sort and get k nearest neighbors (smallest distances)
    dists.sort(axis=1)
    mean_dist_kNN = dists[:, :k_neighbors].mean(axis=1)
    
    return mean_dist_kNN


def get_probability_cols(preds_df: pd.DataFrame, model_name: str):
    """Extract probability columns from PyCaret result."""
    cols = preds_df.columns
    
    if 'prediction_score_0' in cols and 'prediction_score_1' in cols:
        return preds_df['prediction_score_0'], preds_df['prediction_score_1']
    
    elif 'Score' in cols and ('Label' in cols or 'prediction_label' in cols):
        label_col = 'prediction_label' if 'prediction_label' in cols else 'Label'
        score_col = 'Score'
        
        def get_probs(row):
            lbl = int(float(row[label_col]))
            score = row[score_col]
            return (1.0 - score, score) if lbl == 1 else (score, 1.0 - score)
        
        probs = preds_df.apply(get_probs, axis=1, result_type='expand')
        return probs[0], probs[1]
    
    raise KeyError(f"Could not extract probabilities. Columns: {list(cols)}")


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def run_virtual_screening():
    """Execute complete Virtual Screening with Applicability Domain."""
    
    print("=" * 80)
    print("   VIRTUAL SCREENING HsDHODH - With Applicability Domain")
    print("=" * 80)
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # -------------------------------------------------------------------------
    # 1. Load model to extract selected features
    # -------------------------------------------------------------------------
    print("\n📦 1. Loading model to extract SELECTED FEATURES...")
    first_model = load_model_safe(MODEL_PATHS[0])
    
    selected_features = get_selected_features_from_pipeline(first_model)
    
    if selected_features is None:
        print("   ❌ ERROR: Could not extract features from model!")
        return None
    
    print(f"   ✅ Model uses {len(selected_features)} features (after multicollinearity removal)")
    print(f"   Example: {selected_features[:5]}... {selected_features[-5:]}")
    
    # -------------------------------------------------------------------------
    # 2. Load Screening data
    # -------------------------------------------------------------------------
    print(f"\n📂 2. Loading SCREENING data...")
    raw_df = pd.read_csv(CSV_PATH)
    print(f"   -> {len(raw_df)} compounds loaded")
    
    # Rename columns bit_X to X (match with model)
    rename_map = {}
    for col in raw_df.columns:
        if col.startswith(FINGERPRINT_PREFIX):
            bit_num = col[len(FINGERPRINT_PREFIX):]
            if bit_num.isdigit():
                rename_map[col] = bit_num
    
    screening_df = raw_df.rename(columns=rename_map)
    print(f"   -> Renamed {len(rename_map)} columns: bit_X -> X")
    
    # -------------------------------------------------------------------------
    # 3. Load Training data and calculate AD
    # -------------------------------------------------------------------------
    print(f"\n📂 3. Loading TRAINING data...")
    tanimoto_dists = None
    
    if TRAIN_DATA_PATH.exists():
        train_df = pd.read_csv(TRAIN_DATA_PATH)
        print(f"   -> {len(train_df)} training compounds loaded")
        
        # Check common features
        features_in_screening = [f for f in selected_features if f in screening_df.columns]
        features_in_train = [f for f in selected_features if f in train_df.columns]
        common_features = [f for f in selected_features if f in features_in_screening and f in features_in_train]
        
        print(f"   -> Features in screening: {len(features_in_screening)}/{len(selected_features)}")
        print(f"   -> Features in training: {len(features_in_train)}/{len(selected_features)}")
        
        if len(common_features) == len(selected_features):
            print(f"   ✅ All {len(selected_features)} features found!")
        else:
            missing = set(selected_features) - set(common_features)
            print(f"   ⚠️ Missing features: {list(missing)[:5]}...")
        
        if len(common_features) > 0:
            # Create matrices with selected features
            query_matrix = screening_df[common_features].values.astype(float)
            train_matrix = train_df[common_features].values.astype(float)
            
            print(f"\n📊 4. Calculating APPLICABILITY DOMAIN...")
            print(f"   -> Using {len(common_features)} selected features")
            
            if query_matrix.sum() > 0 and train_matrix.sum() > 0:
                tanimoto_dists = calculate_ad_tanimoto(query_matrix, train_matrix, K_NEIGHBORS)
                print(f"   ✅ Distances calculated!")
                print(f"   -> Range: [{tanimoto_dists.min():.4f}, {tanimoto_dists.max():.4f}]")
                print(f"   -> Mean: {tanimoto_dists.mean():.4f}")
            else:
                print("   ⚠️ WARNING: Matrices contain only zeros!")
    else:
        print(f"   ⚠️ WARNING: Training data not found!")

    # -------------------------------------------------------------------------
    # 5. Initialize results
    # -------------------------------------------------------------------------
    results = pd.DataFrame()
    results['Index_Original'] = raw_df.index
    
    if SMILES_COL in raw_df.columns: 
        results[SMILES_COL] = raw_df[SMILES_COL].values
    
    if MOL_ID_COL in raw_df.columns:
        results[MOL_ID_COL] = raw_df[MOL_ID_COL].values
    else:
        results[MOL_ID_COL] = [f"CMP_{i}" for i in raw_df.index]

    # Add AD results
    if tanimoto_dists is not None:
        results['AD_Tanimoto_Dist'] = tanimoto_dists
        # Round AD distances to 2 decimal places (standard rounding)
        results['AD_Tanimoto_Dist'] = results['AD_Tanimoto_Dist'].round(2)
        results['AD_Inside'] = results['AD_Tanimoto_Dist'] < TANIMOTO_THRESHOLD

    # -------------------------------------------------------------------------
    # 6. Model predictions
    # -------------------------------------------------------------------------
    print(f"\n🤖 5. Running PREDICTIONS...")
    model_names = []
    
    for model_path in MODEL_PATHS:
        model_name = "RF" if "RandomForest" in model_path.name else ("XGB" if "XGB" in model_path.name else model_path.stem[:5])
        model_names.append(model_name)
        
        print(f"   --> {model_name}...")
        model = load_model_safe(model_path)
        
        # Prepare data (remove Outcome if exists)
        pred_data = screening_df.copy()
        if 'Outcome' in pred_data.columns:
            pred_data = pred_data.drop(columns=['Outcome'])
        
        # Prediction
        preds = predict_model(model, data=pred_data, raw_score=True)
        prob_0, prob_1 = get_probability_cols(preds, model_name)
        
        results[f'proba_{model_name}'] = prob_1.values
        results[f'pred_{model_name}'] = (prob_1 >= 0.5).astype(int).values

    # -------------------------------------------------------------------------
    # 7. Consensus and final status
    # -------------------------------------------------------------------------
    proba_cols = [c for c in results.columns if c.startswith('proba_') and not c.endswith('consensus')]
    results['proba_consensus'] = results[proba_cols].mean(axis=1)
    # Round probabilities to two decimals (standard rounding)
    for _c in proba_cols + ['proba_consensus']:
        if _c in results.columns:
            # Ensure proba_RF and proba_XGB never round to 1.00; cap at 0.99 before rounding
            if _c in ['proba_RF', 'proba_XGB']:
                results[_c] = results[_c].clip(upper=0.99).round(2)
            else:
                results[_c] = results[_c].round(2)

    results['pred_consensus'] = (results['proba_consensus'] >= 0.5).astype(int)
    
    # AD Status (removed)
    
    # Sort and rank
    results = results.sort_values('proba_consensus', ascending=False).reset_index(drop=True)
    results['Rank'] = results.index + 1
    
    # Reorder columns
    first_cols = ['Rank', MOL_ID_COL]
    if SMILES_COL in results.columns:
        first_cols.append(SMILES_COL)
    first_cols.extend(['proba_consensus', 'pred_consensus'])
    if 'AD_Tanimoto_Dist' in results.columns:
        first_cols.extend(['AD_Tanimoto_Dist'])
    
    other_cols = [c for c in results.columns if c not in first_cols]
    results = results[first_cols + other_cols]
    
    # -------------------------------------------------------------------------
    # 8. Save results
    # -------------------------------------------------------------------------
    timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = OUTPUT_DIR / f'VS_Results_AD_{timestamp}.csv'
    results.to_csv(output_path, index=False)
    
    # -------------------------------------------------------------------------
    # 9. Summary
    # -------------------------------------------------------------------------
    print(f"\n" + "=" * 80)
    print(f"✅ COMPLETED! Results saved to:")
    print(f"   {output_path}")
    print("=" * 80)
    
    # Preview
    preview_cols = ['Rank', MOL_ID_COL, 'proba_consensus']
    if 'AD_Tanimoto_Dist' in results.columns:
        preview_cols.extend(['AD_Tanimoto_Dist'])
    
    print(f"\n📊 TOP 10 COMPOUNDS:")
    print(results[preview_cols].head(10).to_string())
    
    # Statistics
    if 'AD_Inside' in results.columns:
        inside = (results['AD_Inside']).sum()
        outside = (~results['AD_Inside']).sum()
        print(f"\n📈 APPLICABILITY DOMAIN SUMMARY:")
        print(f"   • Inside Domain: {inside} compounds")
        print(f"   • Outside Domain: {outside} compounds")
        print(f"   • Threshold: {TANIMOTO_THRESHOLD} (distance)")
    
    actives = (results['pred_consensus'] == 1).sum()
    print(f"\n📈 PREDICTION SUMMARY:")
    print(f"   • Predicted as Active: {actives}")
    print(f"   • Predicted as Inactive: {len(results) - actives}")
    
    return results


# Run
VS_RESULTS = run_virtual_screening()

   VIRTUAL SCREENING HsDHODH - With Applicability Domain

📦 1. Loading model to extract SELECTED FEATURES...
Transformation Pipeline and Model Successfully Loaded
   ✅ Model uses 1845 features (after multicollinearity removal)
   Example: ['0', '1', '2', '3', '4']... ['2041', '2042', '2043', '2044', '2047']

📂 2. Loading SCREENING data...
   -> 10 compounds loaded
   -> Renamed 2048 columns: bit_X -> X

📂 3. Loading TRAINING data...
   -> 1792 training compounds loaded
   -> Features in screening: 1845/1845
   -> Features in training: 1845/1845
   ✅ All 1845 features found!

📊 4. Calculating APPLICABILITY DOMAIN...
   -> Using 1845 selected features
   -> Calculating Tanimoto distance (Query: (10, 1845), Train: (1792, 1845))...
   ✅ Distances calculated!
   -> Range: [0.6505, 0.7526]
   -> Mean: 0.6928

🤖 5. Running PREDICTIONS...
   --> RF...
Transformation Pipeline and Model Successfully Loaded
   --> XGB...
Transformation Pipeline and Model Successfully Loaded

✅ COMPLETED! Results 

## 3. Interpretability with SHAP (Optional)

This section uses **SHAP (SHapley Additive exPlanations)** to explain model predictions.

SHAP generates force plots showing which fingerprint bits contributed most to each prediction, helping to understand the chemical features driving activity predictions.

⚠️ **Note:** This section requires the Virtual Screening cell to be executed first (`VS_RESULTS` must exist in kernel).

In [11]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import warnings

# Ignorar avisos de depreciação do SHAP/NumPy para limpar a saída
warnings.filterwarnings("ignore")

# ============================================================================
# 3. INTERPRETABILITY WITH SHAP (NORMALIZED TO PROBABILITY)
# ============================================================================

print("=" * 80)
print("   INTERPRETABILITY ANALYSIS - NORMALIZED PROBABILITIES (0-1)")
print("=" * 80)

# Diretórios para salvar os outputs
SHAP_BASE_DIR = BASE_PATH / "output" / "shap"
HTML_DIR = SHAP_BASE_DIR / "html"
PNG_DIR = SHAP_BASE_DIR / "png"

for d in [SHAP_BASE_DIR, HTML_DIR, PNG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 1. Carregar dados
print("\n📂 Reloading screening data...")
raw_df_shap = pd.read_csv(CSV_PATH)

# Mapear colunas bit_X -> X
rename_map_shap = {}
for col in raw_df_shap.columns:
    if col.startswith(FINGERPRINT_PREFIX):
        bit_num = col[len(FINGERPRINT_PREFIX):]
        if bit_num.isdigit():
            rename_map_shap[col] = bit_num

X_all = raw_df_shap.rename(columns=rename_map_shap)

# Identificar coluna ID
possible_id_cols = [c for c in raw_df_shap.columns if c.lower() == 'id']
id_col = possible_id_cols[0] if possible_id_cols else ('Title' if 'Title' in raw_df_shap.columns else raw_df_shap.columns[0])
print(f"   -> Using ID column: '{id_col}'")

# Preparar Background Data para o SHAP (Necessário para converter log-odds em probabilidade)
# Como o dataset de triagem é pequeno, usamos ele todo. Se fosse enorme, usaríamos shap.sample(X_all, 100)
X_background = X_all

# 2. Iterar sobre os modelos
for model_path in MODEL_PATHS:
    model_name = "RF" if "RandomForest" in model_path.name else ("XGB" if "XGB" in model_path.name else model_path.stem[:5])
    print(f"\n🤖 Analyzing Model: {model_name}...")
    
    # Carregar modelo
    model = load_model_safe(model_path)
    if hasattr(model, 'steps'):
        final_estimator = model.steps[-1][1]
    else:
        final_estimator = model
        
    feature_names = get_selected_features_from_pipeline(model)
    if feature_names is None: continue

    # Filtrar features
    try:
        X_model = X_all[feature_names].astype(float)
        X_bg_model = X_background[feature_names].astype(float)
    except KeyError as e:
        print(f"   ❌ Missing features: {e}")
        continue

    # --- MUDANÇA PRINCIPAL: CALCULAR EM PROBABILIDADE ---
    print("   -> Configuring Explainer for PROBABILITY output (0-1)...")
    
    # model_output="probability" força a conversão. 
    # data=X_bg_model é obrigatório para isso funcionar em alguns modelos.
    try:
        explainer = shap.TreeExplainer(
            final_estimator, 
            data=X_bg_model, 
            feature_perturbation="interventional",
            model_output="probability"
        )
        
        # Calcular SHAP values (agora na escala de probabilidade)
        # Nota: check_additivity=False pode ser necessário devido a arredondamentos de precisão na conversão prob
        shap_values_prob = explainer.shap_values(X_model, check_additivity=False)
        
        # Obter o expected_value (Base Value) em probabilidade
        expected_value_prob = explainer.expected_value
        
    except Exception as e:
        print(f"   ⚠️ Could not force probability output directly ({e}). Falling back to default.")
        # Fallback para padrão
        explainer = shap.TreeExplainer(final_estimator)
        shap_values_prob = explainer.shap_values(X_model)
        expected_value_prob = explainer.expected_value

    # Ajuste de dimensões (RF e XGBoost com probability retornam classes separadas)
    # Geralmente: [previsão_classe_0, previsão_classe_1]
    target_class_idx = 1 # Classe 1 = Ativo
    
    if isinstance(shap_values_prob, list):
        # Random Forest geralmente retorna lista de arrays
        shap_vals_target = shap_values_prob[target_class_idx]
        base_val_target = expected_value_prob[target_class_idx]
        
    elif isinstance(shap_values_prob, np.ndarray) and shap_values_prob.ndim == 3:
        # XGBoost com output="probability" retorna array 3D [samples, features, classes]
        shap_vals_target = shap_values_prob[:, :, target_class_idx]
        # O expected_value pode ser array ou escalar dependendo da versão
        if isinstance(expected_value_prob, np.ndarray) or isinstance(expected_value_prob, list):
             base_val_target = expected_value_prob[target_class_idx]
        else:
             base_val_target = expected_value_prob
    else:
        # Caso fallback (ex: regressão ou binário simples)
        shap_vals_target = shap_values_prob
        base_val_target = expected_value_prob

    # 3. Gerar Gráficos
    print(f"   -> Generating plots for {len(X_model)} compounds...")
    
    for i in range(len(X_model)):
        mol_id = str(raw_df_shap.iloc[i][id_col])
        safe_mol_id = "".join([c for c in mol_id if c.isalnum() or c in ('-','_','.')]).strip()
        
        features_val = X_model.iloc[i]
        
        # Validar f(x) final para este composto
        final_prob = base_val_target + np.sum(shap_vals_target[i])
        
        # --- A. PNG: FORCE PLOT (Normalizado) ---
        try:
            plt.figure(figsize=(25, 6))
            shap.force_plot(
                base_val_target,
                shap_vals_target[i],
                features_val,
                feature_names=feature_names,
                matplotlib=True,
                show=False,
                text_rotation=45,
                contribution_threshold=0.02 # Mostrar apenas contribuições > 2%
            )
            
            # Título informativo com a probabilidade final
            plt.title(f"Force Plot: {model_name} - {mol_id}\nFinal Probability f(x) = {final_prob:.4f}", 
                      fontsize=16, loc='right', y=1.5, fontweight='bold')
            
            png_filename = PNG_DIR / f"{model_name}_{safe_mol_id}_force_prob.png"
            plt.savefig(png_filename, dpi=300, bbox_inches='tight')
            plt.close()
            
        except Exception as e:
            print(f"      ⚠️ Error PNG {safe_mol_id}: {e}")
            plt.close()

        # --- B. HTML: FORCE PLOT ---
        try:
            p = shap.force_plot(
                base_val_target,
                shap_vals_target[i],
                features_val,
                feature_names=feature_names,
                matplotlib=False,
                link="identity" # Já convertemos para probabilidade, não precisa de 'logit'
            )
            html_filename = HTML_DIR / f"{model_name}_{safe_mol_id}_force_prob.html"
            shap.save_html(str(html_filename), p)
        except Exception as e: pass

    print(f"   ✅ Plots saved for {model_name}")

print("\n✅ Analysis Completed. Values are now probabilities (0.0 to 1.0).")

   INTERPRETABILITY ANALYSIS - NORMALIZED PROBABILITIES (0-1)

📂 Reloading screening data...
   -> Using ID column: 'ID'

🤖 Analyzing Model: RF...
Transformation Pipeline and Model Successfully Loaded
   -> Configuring Explainer for PROBABILITY output (0-1)...
   -> Generating plots for 10 compounds...
   ✅ Plots saved for RF

🤖 Analyzing Model: XGB...
Transformation Pipeline and Model Successfully Loaded
   -> Configuring Explainer for PROBABILITY output (0-1)...
   -> Generating plots for 10 compounds...
   ✅ Plots saved for XGB

✅ Analysis Completed. Values are now probabilities (0.0 to 1.0).


### 3.2 SHAP Summary Plot (Global Feature Importance)

This cell generates a summary plot showing the overall importance of each fingerprint bit across all predictions.

In [19]:
import os
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ==========================================
# 1. SETUP DIRECTORY
# ==========================================
output_dir = r"C:\Users\sabri\repositorios\Identification-of-Potent-HsDHODH-Inhibitors-via-Integrated-Computational-Pipeline\output\shap"
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Output directory verified: {output_dir}")

# ==========================================
# 2. ITERATE AND GENERATE PLOTS
# ==========================================
# (Assuming MODEL_PATHS, load_model_safe, X_all, etc. are already defined in previous cells)

for model_path in MODEL_PATHS:
    # --- Your Existing Loading Logic ---
    model_name = "RF" if "RandomForest" in model_path.name else ("XGB" if "XGB" in model_path.name else model_path.stem[:5])
    print(f"\n🤖 Analyzing Model: {model_name}...")

    # Load model
    model = load_model_safe(model_path)
    
    # Extract the actual estimator from pipeline if necessary
    if hasattr(model, 'steps'):
        final_estimator = model.steps[-1][1]
    else:
        final_estimator = model

    # Get feature names
    feature_names = get_selected_features_from_pipeline(model)
    if feature_names is None: 
        print("   ⚠️ No feature names found, skipping.")
        continue

    # Filter features (Prepare Data)
    try:
        # Ensure we are working with a DataFrame for SHAP to see column names
        X_model = X_all[feature_names].astype(float)
        # Check if X_background is needed; usually TreeExplainer works fine with just X_model for RF/XGB
        # If your dataset is huge, consider taking a sample: X_model_sample = X_model.sample(100)
    except KeyError as e:
        print(f"   ❌ Missing features: {e}")
        continue

    # --- SHAP GENERATION LOGIC (Added Here) ---
    try:
        print(f"   🔬 Generating SHAP values for {model_name}...")
        
        # 1. Initialize Explainer
        # TreeExplainer is specific for RF and XGB
        explainer = shap.TreeExplainer(final_estimator)
        
        # 2. Calculate SHAP values
        # We use X_model (the filtered features for this specific model)
        shap_values = explainer.shap_values(X_model)

        # 3. Create Plot
        plt.figure(figsize=(10, 6))
        
        # Handle Binary Classification (returns list of [Class0, Class1])
        if isinstance(shap_values, list):
            # Plot Class 1 (Active/Positive)
            shap.summary_plot(shap_values[1], X_model, show=False)
        else:
            shap.summary_plot(shap_values, X_model, show=False)

        # 4. Save Plot
        filename = f"shap_summary_importance_{model_name}.png"
        save_path = os.path.join(output_dir, filename)
        
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close() # Important to close memory
        
        print(f"   ✅ Plot saved to: {save_path}")

    except Exception as e:
        print(f"   ❌ SHAP Error for {model_name}: {e}")

print("\n🏁 Process Complete.")

✅ Output directory verified: C:\Users\sabri\repositorios\Identification-of-Potent-HsDHODH-Inhibitors-via-Integrated-Computational-Pipeline\output\shap

🤖 Analyzing Model: RF...
Transformation Pipeline and Model Successfully Loaded
   🔬 Generating SHAP values for RF...
   ✅ Plot saved to: C:\Users\sabri\repositorios\Identification-of-Potent-HsDHODH-Inhibitors-via-Integrated-Computational-Pipeline\output\shap\shap_summary_importance_RF.png

🤖 Analyzing Model: XGB...
Transformation Pipeline and Model Successfully Loaded
   🔬 Generating SHAP values for XGB...
   ✅ Plot saved to: C:\Users\sabri\repositorios\Identification-of-Potent-HsDHODH-Inhibitors-via-Integrated-Computational-Pipeline\output\shap\shap_summary_importance_XGB.png

🏁 Process Complete.
